使用

In [1]:
from dotenv import load_dotenv
from langchain_deepseek.chat_models import ChatDeepSeek
from langchain.agents import create_agent
import os

load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 创建DeepSeek LLM
model = ChatDeepSeek(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model="deepseek-flash"
)

agent = create_agent(model)

# 调用
response= agent.invoke({
    "messages":[
        {"role":"system","content":"你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题"},
        {"role":"user","content":"100 + 20 * 3 = ？"}
    ]
})

from rich import print as rprint
rprint(response)

{
    'messages': [
        SystemMessage(
            content='你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题',
            additional_kwargs={},
            response_metadata={},
            id='9e322f2d-5bfe-4129-b34c-7c8a99c1ac19'
        ),
        HumanMessage(
            content='100 + 20 * 3 = ？',
            additional_kwargs={},
            response_metadata={},
            id='2df0a252-b1aa-49be-944a-96f30882398b'
        ),
        AIMessage(
            content='**160**\n\n因为先算乘法：\\(20 \\times 3 = 60\\)  \n再算加法：\\(100 + 60 = 160\\)。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '我们需要回答中文。用户问 100 + 20 * 3 = ? 数学运算优先级。需要给结果。100 + 
60 = 160。需要可能简要说明先乘后加。用中文。确保不犯错。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 83,
                    'prompt_tokens': 54,
                    'total_tokens': 137,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 49,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 54
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '6c809358-40b8-4b7d-b98b-8ce0d34cc895',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a14584-5b4b-72c3-b1cf-a8d53c65d67a-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 54,
                'output_tokens': 83,
                'total_tokens': 137,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 49}
            }
        )
    ]
}

绑定工具

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv(override=True)

try_cnt = 0

@tool(parse_docstring=True)
def get_weather(city: str):
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    global try_cnt
    try_cnt += 1
    if try_cnt < 3:
        return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"

    return f"{city}今天天气挺好"

@tool(parse_docstring=True)
def get_news():
    """
    新闻查询工具
    """
    return "近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。"

agent = create_agent(
    model,
    tools=[get_weather, get_news]
)

response = agent.invoke({
    "messages": [
        SystemMessage("""
当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，
说明是临时故障，不要立即放弃；
你应再次调用同一个工具，最多重试 3 次。
如果 3 次后仍失败，再向用户说明服务暂时不可用。
"""),
        HumanMessage("你好，杭州今天的天气如何？今天有哪些新闻？"),
    ]
})
rprint(response)


{
    'messages': [
        SystemMessage(
            content="\n当工具返回以 'TEMP_UNAVAILABLE:' 
开头的结果时，\n说明是临时故障，不要立即放弃；\n你应再次调用同一个工具，最多重试 3 次。\n如果 3 
次后仍失败，再向用户说明服务暂时不可用。\n",
            additional_kwargs={},
            response_metadata={},
            id='5d21bb45-ca41-423a-884c-3970d902fb46'
        ),
        HumanMessage(
            content='你好，杭州今天的天气如何？今天有哪些新闻？',
            additional_kwargs={},
            response_metadata={},
            id='05e9c93a-44a2-42db-a741-6c32e1fb39e8'
        ),
        AIMessage(
            content='你好！我来帮你查询杭州的天气和今天的新闻。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': "The user wants Hangzhou weather and today's news. I should call both tools in
parallel."
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 82,
                    'prompt_tokens': 393,
                    'total_tokens': 475,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 19,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 137
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '90865625-4cfb-4036-a18d-276117b047af',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a14590-9d03-7673-b233-f48a9db808c1-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '杭州'},
                    'id': 'call_00_bJCT5TUBef9QR9SPYNjx4590',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_01_AHSMPWjQJePYemuJhhtd3282', 'type': 'tool_call'}
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 393,
                'output_tokens': 82,
                'total_tokens': 475,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {'reasoning': 19}
            }
        ),
        ToolMessage(
            content='TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试',
            name='get_weather',
            id='dc5ab89b-a082-4e3f-8636-d933c6e5caf8',
            tool_call_id='call_00_bJCT5TUBef9QR9SPYNjx4590'
        ),
        ToolMessage(
            content='近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍
上涨，旧手机成“香饽饽”。',
            name='get_news',
            id='3d175425-fdc2-436b-8087-37d4d46230c5',
            tool_call_id='call_01_AHSMPWjQJePYemuJhhtd3282'
        ),
        AIMessage(
            content='天气服务返回了临时故障，我重试一下。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 'Weather tool returned TEMP_UNAVAILABLE. I should retry up to 3 times.'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 70,
                    'prompt_tokens': 553,
                    'total_tokens': 623,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 20,
                        'rejected_pred